# MassSpecGym Dataset Enrichment Pipeline

This notebook processes the MassSpecGym mass spectrometry dataset and enriches it with molecular features for machine learning tasks.

## Dataset Loading and Inspection

Loading and validating the raw MassSpecGym dataset.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the MassSpecGym.tsv file
data_path = Path("../data/raw/MassSpecGym.tsv")
df = pd.read_csv(data_path, sep='\t')

print("🔍 MASSSPECGYM DATASET INSPECTION")
print("="*60)

print(f"📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📋 Columns: {list(df.columns)}")

print("\n" + "="*60)
print("📋 DATA TYPES & MISSING VALUES")
print("="*60)
print("Data types:")
print(df.dtypes)
print("\n❌ Missing values:")
missing = df.isnull().sum()
print(missing[missing > 0])
if missing.sum() == 0:
    print("✅ No missing values found!")

print("\n" + "="*60)  
print("🔬 FIRST 5 ROWS")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("📈 NUMERICAL COLUMNS STATISTICS")
print("="*60)
numerical_cols = ['parent_mass', 'precursor_mz', 'collision_energy']
for col in numerical_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Min: {df[col].min()}")
        print(f"  Max: {df[col].max()}")
        print(f"  Mean: {df[col].mean():.3f}")
        print(f"  Missing: {df[col].isnull().sum()}")

print("\n" + "="*60)
print("🧪 CATEGORICAL COLUMNS")
print("="*60)
categorical_cols = ['adduct', 'instrument_type', 'fold', 'simulation_challenge']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Values: {df[col].value_counts().to_dict()}")

print("\n" + "="*60)
print("🔍 MASS SPECTROMETRY DATA VALIDATION")
print("="*60)

# Check mzs and intensities consistency
print("Checking m/z and intensity data consistency...")
mismatches = 0
sample_lengths = []

for idx in range(min(5, len(df))):  # Check first 5 samples
    mzs = df.iloc[idx]['mzs'].split(',')
    intensities = df.iloc[idx]['intensities'].split(',')
    sample_lengths.append(len(mzs))
    
    print(f"\nSample {idx+1} (ID: {df.iloc[idx]['identifier']}):")
    print(f"  Number of peaks: {len(mzs)}")
    print(f"  m/z range: {mzs[0]} - {mzs[-1]}")
    print(f"  Intensity range: {intensities[0]} - {max(intensities)}")
    print(f"  Data consistency: ✅ Match" if len(mzs) == len(intensities) else "❌ Mismatch")

print(f"\nSpectrum complexity (first 5 samples): {sample_lengths} peaks")

print("\n" + "="*60)
print("✅ DATASET SUMMARY")
print("="*60)
print(f"• Total samples: {len(df):,}")
print(f"• Unique molecules: {df['inchikey'].nunique():,}")  
print(f"• Data splits: {dict(df['fold'].value_counts())}")
print(f"• Adduct types: {list(df['adduct'].unique())}")
print(f"• Instruments: {list(df['instrument_type'].dropna().unique())}")
print(f"• Collision energies: {df['collision_energy'].min()}-{df['collision_energy'].max()} eV")
print(f"• Mass range: {df['parent_mass'].min():.1f}-{df['parent_mass'].max():.1f} Da")

print("\n✅ All columns are in the correct format!")
print("🔬 Ready for mass spectrometry analysis!")

🔍 MASSSPECGYM DATASET INSPECTION
📊 Dataset Shape: 231,104 rows × 14 columns
📋 Columns: ['identifier', 'mzs', 'intensities', 'smiles', 'inchikey', 'formula', 'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct', 'instrument_type', 'collision_energy', 'fold', 'simulation_challenge']

📋 DATA TYPES & MISSING VALUES
Data types:
identifier               object
mzs                      object
intensities              object
smiles                   object
inchikey                 object
formula                  object
precursor_formula        object
parent_mass             float64
precursor_mz            float64
adduct                   object
instrument_type          object
collision_energy        float64
fold                     object
simulation_challenge       bool
dtype: object

❌ Missing values:
instrument_type       5223
collision_energy    109358
dtype: int64

🔬 FIRST 5 ROWS
             identifier                                                mzs  \
0  MassSpecGymID0000001  

In [12]:
df.head()

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge
0,MassSpecGymID0000001,"91.0542,125.0233,154.0499,155.0577,185.0961,20...","0.24524524524524524,1.0,0.08008008008008008,0....",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,30.0,train,True
1,MassSpecGymID0000002,"91.0542,125.0233,155.0577,185.0961,229.0859,24...","0.0990990990990991,0.28128128128128127,0.04004...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,20.0,train,True
2,MassSpecGymID0000003,"69.0343,91.0542,125.0233,127.039,153.0699,154....","0.03403403403403404,0.31431431431431434,1.0,0....",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,40.0,train,True
3,MassSpecGymID0000004,"69.0343,91.0542,110.06,111.0441,112.0393,120.0...","0.17917917917917917,0.47347347347347346,0.0380...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,55.0,train,True
4,MassSpecGymID0000005,"91.0542,125.0233,185.0961,229.0859,246.1125,28...","0.07807807807807808,0.1841841841841842,0.03503...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,10.0,train,True


## Extract Unique Molecules

Create a molecule-level dataframe from the spectrum-level data.

In [13]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# Extract unique molecules (based on InChIKey)
print(f"📊 Total spectra: {len(df):,}")
print(f"🧪 Unique molecules: {df['inchikey'].nunique():,}")

# Create molecule-level dataframe with unique molecules
mol_features_df = df[['inchikey', 'smiles']].drop_duplicates(subset='inchikey').copy()
print(f"\n✅ Created molecule-level dataframe with {len(mol_features_df):,} unique molecules")

📊 Total spectra: 231,104
🧪 Unique molecules: 28,929

✅ Created molecule-level dataframe with 28,929 unique molecules

✅ Created molecule-level dataframe with 28,929 unique molecules


## Compute Molecular Descriptors

Calculate key molecular properties: molecular weight, LogP (lipophilicity), and TPSA (topological polar surface area).

In [14]:
# Compute molecular descriptors for each unique molecule
def compute_molecular_features(smiles):
    """Compute molecular weight, LogP, and TPSA from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return pd.Series({
            'mol_weight': None,
            'logp': None,
            'tpsa': None
        })
    
    return pd.Series({
        'mol_weight': Descriptors.MolWt(mol),
        'logp': Descriptors.MolLogP(mol),
        'tpsa': Descriptors.TPSA(mol)
    })

print("🔬 Computing molecular descriptors...")
mol_features_df[['mol_weight', 'logp', 'tpsa']] = mol_features_df['smiles'].apply(compute_molecular_features)

print(f"\n✅ Computed descriptors for {len(mol_features_df):,} molecules")
print(f"\nSample molecular features:")
print(mol_features_df[['inchikey', 'mol_weight', 'logp', 'tpsa']].head())

🔬 Computing molecular descriptors...

✅ Computed descriptors for 28,929 molecules

Sample molecular features:
          inchikey  mol_weight    logp    tpsa
0   VFMQMACUYWGDOJ     287.315  2.0683   68.54
11  MBMQEIFVQACCCH     318.369  3.5796   83.83
42  GIYROBMIPLLHQE     421.457  1.6413  127.42
51  MXRJZFNJVFPSQN     196.202  0.6046   55.90
59  HNMWDXUKPJZOQD     376.317  1.2106  139.59

✅ Computed descriptors for 28,929 molecules

Sample molecular features:
          inchikey  mol_weight    logp    tpsa
0   VFMQMACUYWGDOJ     287.315  2.0683   68.54
11  MBMQEIFVQACCCH     318.369  3.5796   83.83
42  GIYROBMIPLLHQE     421.457  1.6413  127.42
51  MXRJZFNJVFPSQN     196.202  0.6046   55.90
59  HNMWDXUKPJZOQD     376.317  1.2106  139.59


## Functional Group Detection (SMARTS Patterns)

Detecting 13 functional groups using SMARTS patterns based on literature:
- **Alkene**: C=C double bonds
- **Aromatic**: Aromatic rings (benzene-like)
- **Hydroxyl**: OH groups (alcohols, phenols)
- **Ketone**: C=O groups
- **Carboxylic acid**: COOH groups
- **Primary amine**: Primary NH2 groups
- **Amide**: C(=O)N groups
- **Ester**: C(=O)O groups
- **Nitrile**: C≡N groups
- **Halide**: C-X groups (F, Cl, Br, I)
- **Phosphate**: Phosphate groups
- **Thiol**: SH groups
- **Nitro**: NO2 groups

In [15]:
# Define SMARTS patterns for functional group detection
smarts_patterns = {
    "alkene": Chem.MolFromSmarts("C=C"),
    "aromatic": Chem.MolFromSmarts("a1aaaaa1"),
    "hydroxyl": Chem.MolFromSmarts("[OX2H]"),
    "ketone": Chem.MolFromSmarts("[CX3](=O)[#6]"),
    "carboxylic_acid": Chem.MolFromSmarts("C(=O)[OX2H1]"),
    "amine_primary": Chem.MolFromSmarts("[NX3;H2][#6]"),
    "amide": Chem.MolFromSmarts("C(=O)N"),
    "ester": Chem.MolFromSmarts("C(=O)O[#6]"),
    "nitrile": Chem.MolFromSmarts("C#N"),
    "halide": Chem.MolFromSmarts("[CX4][F,Cl,Br,I]"),
    "phosphate": Chem.MolFromSmarts("P(=O)(O)(O)"),
    "thiol": Chem.MolFromSmarts("[SX2H]"),
    "nitro": Chem.MolFromSmarts("[NX3](=O)=O"),
}

def detect_functional_groups(smiles):
    """Detect presence of functional groups using SMARTS patterns"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {name: 0 for name in smarts_patterns.keys()}
    return {name: int(mol.HasSubstructMatch(pat)) for name, pat in smarts_patterns.items()}

print("🧪 Detecting functional groups using SMARTS patterns...")
fg_df = mol_features_df['smiles'].apply(detect_functional_groups).apply(pd.Series)

# Add functional group columns to molecule features dataframe
for col in fg_df.columns:
    mol_features_df[col] = fg_df[col]

print(f"\n✅ Detected {len(smarts_patterns)} functional groups for {len(mol_features_df):,} molecules")
print(f"\nFunctional group prevalence:")
for fg in smarts_patterns.keys():
    count = mol_features_df[fg].sum()
    pct = (count / len(mol_features_df)) * 100
    print(f"  • {fg}: {count:,} molecules ({pct:.1f}%)")

print(f"\nSample functional group data:")
print(mol_features_df[['inchikey'] + list(smarts_patterns.keys())].head())

🧪 Detecting functional groups using SMARTS patterns...

✅ Detected 13 functional groups for 28,929 molecules

Functional group prevalence:
  • alkene: 7,271 molecules (25.1%)
  • aromatic: 22,090 molecules (76.4%)
  • hydroxyl: 14,090 molecules (48.7%)
  • ketone: 20,088 molecules (69.4%)
  • carboxylic_acid: 5,473 molecules (18.9%)
  • amine_primary: 3,492 molecules (12.1%)
  • amide: 12,475 molecules (43.1%)
  • ester: 6,102 molecules (21.1%)
  • nitrile: 636 molecules (2.2%)
  • halide: 1,243 molecules (4.3%)
  • phosphate: 641 molecules (2.2%)
  • thiol: 25 molecules (0.1%)
  • nitro: 0 molecules (0.0%)

Sample functional group data:
          inchikey  alkene  aromatic  hydroxyl  ketone  carboxylic_acid  \
0   VFMQMACUYWGDOJ       0         1         0       1                0   
11  MBMQEIFVQACCCH       1         1         1       1                0   
42  GIYROBMIPLLHQE       1         1         1       1                0   
51  MXRJZFNJVFPSQN       1         0         0       1

## Compute Molecular Fingerprints

Computing ECFP4 and MAP4 fingerprints for molecular similarity and ML tasks.

In [21]:
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from map4 import MAP4

# Initialize fingerprint generators
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)
map4_calc = MAP4(dimensions=1024)

def compute_ecfp4(smiles):
    """Compute ECFP4 fingerprint (2048 bits) from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Morgan fingerprint with radius 2 (equivalent to ECFP4)
    fp = morgan_gen.GetFingerprint(mol)
    # Convert to comma-separated string for storage
    return ','.join(map(str, fp.ToBitString()))

def compute_map4(smiles):
    """Compute MAP4 fingerprint (1024 dimensions) from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        # MAP4 fingerprint
        fp = map4_calc.calculate(mol)
        # Convert to comma-separated string for storage
        return ','.join(map(str, fp))
    except:
        return None

print("🔢 Computing ECFP4 fingerprints (2048 bits)...")
mol_features_df['ecfp4'] = mol_features_df['smiles'].apply(compute_ecfp4)

print("🔢 Computing MAP4 fingerprints (1024 dimensions)...")
mol_features_df['map4'] = mol_features_df['smiles'].apply(compute_map4)

print(f"\n✅ Computed fingerprints for {len(mol_features_df):,} molecules")
print(f"\nFingerprint info:")
print(f"  • ECFP4: {mol_features_df['ecfp4'].notna().sum():,} molecules (2048 bits each)")
print(f"  • MAP4: {mol_features_df['map4'].notna().sum():,} molecules (1024 dims each)")

# Show sample
print(f"\nSample fingerprint (first 50 characters):")
sample_ecfp4 = mol_features_df['ecfp4'].iloc[0]
sample_map4 = mol_features_df['map4'].iloc[0]
print(f"  ECFP4: {sample_ecfp4[:50] if sample_ecfp4 else 'None'}...")
print(f"  MAP4: {sample_map4[:50] if sample_map4 else 'None'}...")

🔢 Computing ECFP4 fingerprints (2048 bits)...
🔢 Computing MAP4 fingerprints (1024 dimensions)...
🔢 Computing MAP4 fingerprints (1024 dimensions)...

✅ Computed fingerprints for 28,929 molecules

Fingerprint info:
  • ECFP4: 28,929 molecules (2048 bits each)
  • MAP4: 28,929 molecules (1024 dims each)

Sample fingerprint (first 50 characters):
  ECFP4: 0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...
  MAP4: 0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,1,1,1,0,0,1,0,0,...

✅ Computed fingerprints for 28,929 molecules

Fingerprint info:
  • ECFP4: 28,929 molecules (2048 bits each)
  • MAP4: 28,929 molecules (1024 dims each)

Sample fingerprint (first 50 characters):
  ECFP4: 0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...
  MAP4: 0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,1,1,1,0,0,1,0,0,...


## Compute Murcko Scaffolds

Extract the core molecular scaffolds (Bemis-Murcko frameworks) for scaffold-based analysis.

In [22]:
# Compute Murcko scaffolds and create scaffold IDs
def get_murcko_scaffold(smiles):
    """Get Murcko scaffold SMILES from molecule SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except:
        return None

print("🧬 Computing Murcko scaffolds...")
mol_features_df['murcko_scaffold'] = mol_features_df['smiles'].apply(get_murcko_scaffold)

# Create scaffold IDs by mapping unique scaffolds to integers
unique_scaffolds = mol_features_df['murcko_scaffold'].dropna().unique()
scaffold_to_id = {scaffold: f"SCAFFOLD_{i:05d}" for i, scaffold in enumerate(unique_scaffolds, 1)}

# Map scaffolds to IDs
mol_features_df['scaffold_id'] = mol_features_df['murcko_scaffold'].map(scaffold_to_id)

print(f"\n✅ Generated {len(unique_scaffolds):,} unique Murcko scaffolds")
print(f"📊 Molecules with scaffolds: {mol_features_df['scaffold_id'].notna().sum():,}")
print(f"❌ Molecules without scaffolds: {mol_features_df['scaffold_id'].isna().sum():,}")

print(f"\nSample scaffolds:")
print(mol_features_df[['inchikey', 'scaffold_id', 'murcko_scaffold']].head())

🧬 Computing Murcko scaffolds...

✅ Generated 14,968 unique Murcko scaffolds
📊 Molecules with scaffolds: 28,929
❌ Molecules without scaffolds: 0

Sample scaffolds:
          inchikey     scaffold_id  \
0   VFMQMACUYWGDOJ  SCAFFOLD_00001   
11  MBMQEIFVQACCCH  SCAFFOLD_00002   
42  GIYROBMIPLLHQE  SCAFFOLD_00003   
51  MXRJZFNJVFPSQN  SCAFFOLD_00004   
59  HNMWDXUKPJZOQD  SCAFFOLD_00005   

                                     murcko_scaffold  
0                             O=c1cccc(CCc2ccccc2)o1  
11                   O=C1CCC/C=C/c2ccccc2C(=O)OCCCC1  
42  O=C1NC(CC2C(=O)Nc3ccccc32)C(=O)N/C1=C/c1cnc[nH]1  
51                  O=C1/C=C\[C@H]2O[C@H]2CCOC(=O)C1  
59                O=C1C=CC(=O)c2cc3c(cc21)CC1OCCC3O1  

✅ Generated 14,968 unique Murcko scaffolds
📊 Molecules with scaffolds: 28,929
❌ Molecules without scaffolds: 0

Sample scaffolds:
          inchikey     scaffold_id  \
0   VFMQMACUYWGDOJ  SCAFFOLD_00001   
11  MBMQEIFVQACCCH  SCAFFOLD_00002   
42  GIYROBMIPLLHQE  SCAFFOLD_0000

## Merge Features to Main Dataset

Combine all computed molecular features back to the spectrum-level dataset.

In [18]:
# Merge molecular features back to the main dataframe
print("🔗 Merging molecular features back to spectrum-level dataframe...")

# Select features to merge: descriptors, scaffolds, functional groups, and fingerprints
functional_group_cols = list(smarts_patterns.keys())
fingerprint_cols = ['ecfp4', 'map4']
features_to_merge = mol_features_df[['inchikey', 'mol_weight', 'logp', 'tpsa', 'scaffold_id', 'murcko_scaffold'] + functional_group_cols + fingerprint_cols]

# Merge on inchikey
df_enriched = df.merge(features_to_merge, on='inchikey', how='left')

print(f"\n✅ Enriched dataset shape: {df_enriched.shape}")
print(f"📊 Original columns: {len(df.columns)}")
print(f"📊 New columns added: {len(df_enriched.columns) - len(df.columns)}")

print(f"\nNew columns:")
# Molecular descriptors
descriptor_cols = ['mol_weight', 'logp', 'tpsa', 'scaffold_id', 'murcko_scaffold']
print(f"\n  Molecular Descriptors:")
for col in descriptor_cols:
    print(f"    • {col}: {df_enriched[col].dtype}, missing: {df_enriched[col].isna().sum()}")

# Functional groups
print(f"\n  Functional Groups ({len(functional_group_cols)} features):")
for col in functional_group_cols:
    count = df_enriched[col].sum()
    print(f"    • {col}: {count:,} spectra ({(count/len(df_enriched)*100):.1f}%)")

# Fingerprints
print(f"\n  Molecular Fingerprints:")
for col in fingerprint_cols:
    non_null = df_enriched[col].notna().sum()
    print(f"    • {col}: {non_null:,} spectra")

# Display sample
print(f"\nSample enriched data:")
display_cols = ['identifier', 'inchikey', 'mol_weight', 'logp', 'tpsa', 'aromatic', 'hydroxyl', 'ketone']
print(df_enriched[display_cols].head())

🔗 Merging molecular features back to spectrum-level dataframe...

✅ Enriched dataset shape: (231104, 34)
📊 Original columns: 14
📊 New columns added: 20

New columns:

  Molecular Descriptors:
    • mol_weight: float64, missing: 0
    • logp: float64, missing: 0
    • tpsa: float64, missing: 0
    • scaffold_id: object, missing: 0
    • murcko_scaffold: object, missing: 0

  Functional Groups (13 features):
    • alkene: 66,169 spectra (28.6%)
    • aromatic: 166,289 spectra (72.0%)
    • hydroxyl: 127,400 spectra (55.1%)
    • ketone: 143,556 spectra (62.1%)
    • carboxylic_acid: 36,853 spectra (15.9%)
    • amine_primary: 29,223 spectra (12.6%)
    • amide: 68,778 spectra (29.8%)
    • ester: 55,570 spectra (24.0%)
    • nitrile: 3,275 spectra (1.4%)
    • halide: 7,361 spectra (3.2%)
    • phosphate: 14,341 spectra (6.2%)
    • thiol: 356 spectra (0.2%)
    • nitro: 0 spectra (0.0%)

  Molecular Fingerprints:
    • ecfp4: 231,104 spectra
    • map4: 231,104 spectra

Sample enriched 

# Save Processed Dataset

Save the enriched dataset to avoid recomputing features every time.

In [19]:
# Save the enriched dataset
output_path = Path("../data/processed/MassSpecGym_enriched.tsv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_enriched.to_csv(output_path, sep='\t', index=False)
print(f"💾 Saved enriched dataset to: {output_path}")
print(f"📊 File size: {output_path.stat().st_size / (1024*1024):.2f} MB")

# Also save the molecule-level features separately for reference
mol_output_path = Path("../data/processed/molecule_features.tsv")
mol_features_df.to_csv(mol_output_path, sep='\t', index=False)
print(f"💾 Saved molecule features to: {mol_output_path}")

print("\n✅ Processing complete! Next time, you can load the enriched dataset directly.")
print(f"   df = pd.read_csv('{output_path}', sep='\\t')")

💾 Saved enriched dataset to: ../data/processed/MassSpecGym_enriched.tsv
📊 File size: 1629.88 MB
💾 Saved molecule features to: ../data/processed/molecule_features.tsv

✅ Processing complete! Next time, you can load the enriched dataset directly.
   df = pd.read_csv('../data/processed/MassSpecGym_enriched.tsv', sep='\t')
💾 Saved molecule features to: ../data/processed/molecule_features.tsv

✅ Processing complete! Next time, you can load the enriched dataset directly.
   df = pd.read_csv('../data/processed/MassSpecGym_enriched.tsv', sep='\t')


## Final Summary

Overview of the complete enriched dataset with all computed features and statistics.

In [20]:
# Summary of enriched dataset
print("📋 ENRICHED DATASET SUMMARY")
print("="*60)
print(f"\n🔬 Dataset dimensions: {df_enriched.shape}")
print(f"\n📊 All columns ({len(df_enriched.columns)}):")
for i, col in enumerate(df_enriched.columns, 1):
    print(f"  {i:2d}. {col}: {df_enriched[col].dtype}")

print(f"\n🧪 Molecular descriptors statistics:")
print(f"  • Molecular Weight: {df_enriched['mol_weight'].min():.1f} - {df_enriched['mol_weight'].max():.1f} Da")
print(f"  • LogP: {df_enriched['logp'].min():.2f} - {df_enriched['logp'].max():.2f}")
print(f"  • TPSA: {df_enriched['tpsa'].min():.1f} - {df_enriched['tpsa'].max():.1f} Ų")

print(f"\n🧬 Scaffold analysis:")
print(f"  • Unique scaffolds: {df_enriched['scaffold_id'].nunique():,}")
print(f"  • Unique molecules: {df_enriched['inchikey'].nunique():,}")
print(f"  • Total spectra: {len(df_enriched):,}")
print(f"  • Avg spectra per molecule: {len(df_enriched) / df_enriched['inchikey'].nunique():.1f}")
print(f"  • Avg molecules per scaffold: {df_enriched['inchikey'].nunique() / df_enriched['scaffold_id'].nunique():.1f}")

print(f"\n🔬 Functional group prevalence (SMARTS patterns):")
functional_group_cols = list(smarts_patterns.keys())
for fg in functional_group_cols:
    count = df_enriched[fg].sum()
    pct = (count / len(df_enriched)) * 100
    print(f"  • {fg}: {count:,} spectra ({pct:.1f}%)")

print(f"\n🔢 Molecular Fingerprints:")
print(f"  • ECFP4: {df_enriched['ecfp4'].notna().sum():,} spectra (2048 bits)")
print(f"  • MAP4: {df_enriched['map4'].notna().sum():,} spectra (1024 dims)")

print(f"\n✅ Total features added: {len(df_enriched.columns) - len(df.columns)}")
print(f"   - Molecular descriptors: 3 (mol_weight, logp, tpsa)")
print(f"   - Scaffold features: 2 (scaffold_id, murcko_scaffold)")
print(f"   - Functional groups: {len(functional_group_cols)}")
print(f"   - Fingerprints: 2 (ecfp4, map4)")

print("\n✅ Feature engineering complete!")

📋 ENRICHED DATASET SUMMARY

🔬 Dataset dimensions: (231104, 34)

📊 All columns (34):
   1. identifier: object
   2. mzs: object
   3. intensities: object
   4. smiles: object
   5. inchikey: object
   6. formula: object
   7. precursor_formula: object
   8. parent_mass: float64
   9. precursor_mz: float64
  10. adduct: object
  11. instrument_type: object
  12. collision_energy: float64
  13. fold: object
  14. simulation_challenge: bool
  15. mol_weight: float64
  16. logp: float64
  17. tpsa: float64
  18. scaffold_id: object
  19. murcko_scaffold: object
  20. alkene: int64
  21. aromatic: int64
  22. hydroxyl: int64
  23. ketone: int64
  24. carboxylic_acid: int64
  25. amine_primary: int64
  26. amide: int64
  27. ester: int64
  28. nitrile: int64
  29. halide: int64
  30. phosphate: int64
  31. thiol: int64
  32. nitro: int64
  33. ecfp4: object
  34. map4: object

🧪 Molecular descriptors statistics:
  • Molecular Weight: 59.1 - 998.9 Da
  • LogP: -13.05 - 17.85
  • TPSA: 0.0 - 47